# GameTheory-9-BackwardInduction (C#)

## Induction arriere et equilibre de sous-jeu parfait (SPE)

**Twin C#** du notebook [GameTheory-9-BackwardInduction](GameTheory-9-BackwardInduction.ipynb) (Python + numpy/matplotlib).
**Marathon parite .NET ⇄ Python** (#4956), suite de [GameTheory-11-BayesianGames-Csharp](GameTheory-11-BayesianGames-Csharp.ipynb).
BCL .NET seule, 0 NuGet, **from-scratch** (Prong B #3801 : comprendre l'algorithme recursif, pas invoquer numpy).

### Objectifs
- Modeliser un **jeu sous forme extensive** (arbre de jeu) en C#
- Implementer l'**induction arriere** (backward induction) pour resoudre un jeu a information parfaite
- Illustrer les **paradoxes classiques** : jeu du mille-pattes (centipede), guerre d'usure (war of attrition), chaine de magasins (chain store)
- Demontrer la notion d'**equilibre de sous-jeu parfait** (Subgame Perfect Equilibrium, Selten 1965)

### Duree estimee : 40 minutes

## 1. Structure de donnees : arbre de jeu

Un jeu sous forme extensive est un **arbre** ou :
- Chaque noeud appartient a un joueur (ou est terminal, ou est un noeud de chance)
- Chaque arete est une **action**
- Les feuilles portent les **gains** (payoffs) de chaque joueur

On represente cela par deux classes simples : `GameNode` et `ExtensiveFormGame`.

In [1]:
using System.Globalization;

// Helper format invariant (Bug marathon #2 : CurrentCulture ne persiste pas entre cells)
static string FI(double x, string fmt = "F2") => x.ToString(fmt, CultureInfo.InvariantCulture);
static string FV(double[] v, string fmt = "F2") => "[" + string.Join(", ", v.Select(x => FI(x, fmt))) + "]";

// Noeud d'un arbre de jeu. player = -1 (terminal), 0 (nature/chance), >= 1 (joueur).
public class GameNode
{
    public string NodeId;
    public int Player;
    public List<string> Actions = new();
    public Dictionary<string, GameNode> Children = new();
    public double[] Payoffs { get; set; }       // non-null uniquement si terminal
    public string Infoset { get; set; }         // ensemble d'information (optionnel)

    public bool IsTerminal() => Player == -1;
    public bool IsChance() => Player == 0;

    public GameNode(string id, int player)
    {
        NodeId = id;
        Player = player;
    }
}

// Jeu sous forme extensive.
public class ExtensiveFormGame
{
    public string Name;
    public int NumPlayers;
    public GameNode Root;
    public Dictionary<string, GameNode> Nodes = new();

    public ExtensiveFormGame(string name, int numPlayers)
    {
        Name = name;
        NumPlayers = numPlayers;
    }

    public void AddNode(GameNode node) { Nodes[node.NodeId] = node; }
    public void SetRoot(GameNode node) { Root = node; AddNode(node); }
}

display("Classes definies : GameNode (noeud), ExtensiveFormGame (arbre de jeu)");

Classes definies : GameNode (noeud), ExtensiveFormGame (arbre de jeu)

## 2. Algorithme d'induction arriere

**Principe** (Zermelo 1913, formalise par Selten 1965) : on resout le jeu en partant des **feuilles** et en remontant vers la racine.

- **Noeud terminal** : retourner les gains.
- **Noeud de chance** (nature) : retourner l'**esperance** ponderee des gains des enfants.
- **Noeud de decision** (joueur `p`) : le joueur choisit l'action qui **maximise son propre gain** ; on enregistre cette action optimale.

Cet algorithme produit un **equilibre de sous-jeu parfait** (SPE) : la strategie est rationnelle dans **chaque** sous-jeu, meme hors du chemin d'equilibre. C'est plus fort que l'equilibre de Nash (qui peut reposer sur des menaces non credibles).

In [2]:
// Resultat d'induction arriere pour un noeud de decision.
public class NodeSolution
{
    public string OptimalAction;
    public double[] EquilibriumPayoffs;
}

// Induction arriere : resout le jeu par recursion depuis la racine.
// Retourne : (solution par noeud de decision, gains d'equilibre globaux).
public static (Dictionary<string, NodeSolution> solution, double[] eqPayoffs) BackwardInduction(ExtensiveFormGame game)
{
    var solution = new Dictionary<string, NodeSolution>();

    double[] Solve(GameNode node)
    {
        if (node.IsTerminal())
            return node.Payoffs;

        if (node.IsChance())
        {
            // Esperance sur les resultats de nature (payoffs egaux a 1/n ici, mais generique)
            var expected = new double[game.NumPlayers];
            double total = 0.0;
            foreach (var kv in node.Children)
            {
                double w = node.Actions.Contains(kv.Key) ? 1.0 / node.Children.Count : 0.0;
                var cp = Solve(kv.Value);
                for (int i = 0; i < game.NumPlayers; i++) expected[i] += w * cp[i];
                total += w;
            }
            return expected;
        }

        // Noeud de decision : le joueur 'node.Player' maximise son propre gain.
        int player = node.Player;
        string bestAction = null;
        double[] bestPayoffs = null;
        double bestValue = double.NegativeInfinity;

        foreach (var action in node.Actions)
        {
            var childPayoffs = Solve(node.Children[action]);
            double playerValue = childPayoffs[player - 1];  // joueurs 1-indexes, tableau 0-indexe
            if (playerValue > bestValue)
            {
                bestValue = playerValue;
                bestAction = action;
                bestPayoffs = childPayoffs;
            }
        }

        solution[node.NodeId] = new NodeSolution { OptimalAction = bestAction, EquilibriumPayoffs = bestPayoffs };
        return bestPayoffs;
    }

    var eq = Solve(game.Root);
    return (solution, eq);
}

// Affiche la solution d'induction arriere.
public static void DisplayBackwardInduction(ExtensiveFormGame game, Dictionary<string, NodeSolution> solution)
{
    var sb = new StringBuilder();
    sb.AppendLine("Solution par induction arriere : " + game.Name);
    sb.AppendLine(new string('=', 60));
    foreach (var kv in solution)
    {
        var node = game.Nodes[kv.Key];
        var sol = kv.Value;
        sb.AppendLine("  Noeud " + kv.Key + " (J" + node.Player + "): joue '" + sol.OptimalAction + "' -> " + FV(sol.EquilibriumPayoffs));
    }
    display(sb.ToString());
}

display("Fonction BackwardInduction definie");

Fonction BackwardInduction definie

## 3. Jeu d'entree sur le marche

Un **entrant** (J1) decide d'entrer (`Enter`) ou de rester hors du marche (`Out`).
S'il entre, l'**incumbent** (J2, monopole en place) choisit entre `Fight` (guerre des prix) et `Accommodate` (accepter).

| Issue | Entrant | Incumbent |
|-------|---------|-----------|
| Out | 0 | 2 |
| Enter puis Fight | -1 | -1 |
| Enter puis Accommodate | 1 | 1 |

**Intuition** : la menace "Fight" est-elle credible ?

In [3]:
// Construction du jeu d'entree.
public static ExtensiveFormGame CreateEntryGame()
{
    var game = new ExtensiveFormGame("Entry Game", 2);

    var outT = new GameNode("out", -1); outT.Payoffs = new double[]{0, 2};
    var fightT = new GameNode("fight", -1); fightT.Payoffs = new double[]{-1, -1};
    var accT = new GameNode("accommodate", -1); accT.Payoffs = new double[]{1, 1};

    var incumbent = new GameNode("incumbent", 2);
    incumbent.Actions = new List<string>{"Fight", "Accommodate"};
    incumbent.Children["Fight"] = fightT;
    incumbent.Children["Accommodate"] = accT;
    incumbent.Infoset = "I2";

    var entrant = new GameNode("entrant", 1);
    entrant.Actions = new List<string>{"Enter", "Out"};
    entrant.Children["Enter"] = incumbent;
    entrant.Children["Out"] = outT;
    entrant.Infoset = "I1";

    game.SetRoot(entrant);
    game.AddNode(incumbent); game.AddNode(outT); game.AddNode(fightT); game.AddNode(accT);
    return game;
}

var entryGame = CreateEntryGame();
var (entrySolution, entryEq) = BackwardInduction(entryGame);
DisplayBackwardInduction(entryGame, entrySolution);
display("Gains a l'equilibre : " + FV(entryEq));
display("Interpretation :");
display("  - Si l'Entrant entre, l'Incumbent prefere Accommoder (1 > -1)");
display("  - Sachant cela, l'Entrant prefere Entrer (1 > 0)");
display("  -> La menace 'Fight' n'est PAS credible (non-SPE). L'entree se produit.");

Solution par induction arriere : Entry Game
  Noeud incumbent (J2): joue 'Accommodate' -> [1.00, 1.00]
  Noeud entrant (J1): joue 'Enter' -> [1.00, 1.00]


Gains a l'equilibre : [1.00, 1.00]

Interpretation :

  - Si l'Entrant entre, l'Incumbent prefere Accommoder (1 > -1)

  - Sachant cela, l'Entrant prefere Entrer (1 > 0)

  -> La menace 'Fight' n'est PAS credible (non-SPE). L'entree se produit.

## 4. Le paradoxe du mille-pattes (Centipede)

Deux joueurs passent ou prennent a tour de role. La cagnotte **grossit** a chaque tour.
Si un joueur prend (`Take`), il empoche la grosse part et l'autre la petite. Si tout le monde passe jusqu'au bout, gain egal.

**Paradoxe** : l'induction arriere dit "Take des le premier tour", mais empiriquement les humains cooperent longtemps. C'est le conflit classique entre rationalite backward et intuition forward.

In [4]:
// Construction du jeu du mille-pattes a n_rounds tours.
public static ExtensiveFormGame CreateCentipedeGame(int nRounds = 6)
{
    var game = new ExtensiveFormGame("Centipede (" + nRounds + " tours)", 2);

    // Gains si "Take" au tour t (player qui prend = grosse cagnotte).
    double[] PayoffsAtRound(int t)
    {
        double bigPile = t + 1;
        double smallPile = Math.Max(0, t - 1);
        int player = (t % 2 == 0) ? 1 : 2;
        return (player == 1) ? new double[]{bigPile, smallPile} : new double[]{smallPile, bigPile};
    }

    // Dernier terminal (si tous passent jusqu'au bout).
    var lastPass = new GameNode("end", -1);
    lastPass.Payoffs = new double[]{nRounds, nRounds};
    game.AddNode(lastPass);

    GameNode nextNode = lastPass;
    for (int t = nRounds - 1; t >= 0; t--)
    {
        int player = (t % 2 == 0) ? 1 : 2;
        var payoffs = PayoffsAtRound(t);
        var takeTerminal = new GameNode("take_" + t, -1);
        takeTerminal.Payoffs = payoffs;
        game.AddNode(takeTerminal);

        var decision = new GameNode("node_" + t, player);
        decision.Actions = new List<string>{"Take", "Pass"};
        decision.Children["Take"] = takeTerminal;
        decision.Children["Pass"] = nextNode;
        game.AddNode(decision);
        nextNode = decision;
    }
    game.SetRoot(nextNode);
    return game;
}

var centipede6 = CreateCentipedeGame(6);
var (centSolution, centEq) = BackwardInduction(centipede6);
DisplayBackwardInduction(centipede6, centSolution);
display("Gains a l'equilibre SPE : " + FV(centEq));
display("Paradoxe : bien que les gains cooperatifs (6, 6) soient plus eleves,");
display("l'equilibre de sous-jeu parfait impose Take des le tour 0 (1, 0).");

Solution par induction arriere : Centipede (6 tours)
  Noeud node_5 (J2): joue 'Take' -> [4.00, 6.00]
  Noeud node_4 (J1): joue 'Take' -> [5.00, 3.00]
  Noeud node_3 (J2): joue 'Take' -> [2.00, 4.00]
  Noeud node_2 (J1): joue 'Take' -> [3.00, 1.00]
  Noeud node_1 (J2): joue 'Take' -> [0.00, 2.00]
  Noeud node_0 (J1): joue 'Take' -> [1.00, 0.00]


Gains a l'equilibre SPE : [1.00, 0.00]

Paradoxe : bien que les gains cooperatifs (6, 6) soient plus eleves,

l'equilibre de sous-jeu parfait impose Take des le tour 0 (1, 0).

## 5. Efficacite et rationalite limitee

Comparons le gain **SPE** (Take immediat) au gain **cooperatif** (Pass jusqu'au bout).
Avec une probabilite d'erreur (rationalite limitee), le comportement change : on peut montrer qu'il devient rationnel de Passer si l'adversaire est susceptible de se tromper.

In [5]:
// Analyse d'efficacite : SPE vs cooperation, en fonction de n_rounds.
var sbEff = new StringBuilder();
sbEff.AppendLine("Efficacite SPE vs Cooperation (Centipede) :");
sbEff.AppendLine(new string('-', 56));
sbEff.AppendLine("  Tours |  SPE (Take t=0) | Cooperatif (fin) | Ratio");
foreach (var n in new[]{2, 4, 6, 8, 10})
{
    var g = CreateCentipedeGame(n);
    var (_, eq) = BackwardInduction(g);
    double speP1 = eq[0];
    double coopP1 = n;  // gain cooperatif du joueur 1
    double ratio = (coopP1 > 0) ? coopP1 / Math.Max(0.001, speP1) : 0;
    sbEff.AppendLine("  " + n.ToString("D5") + " |  " + FI(speP1, "F1") + "          |  " + FI(coopP1, "F1") + "            | " + FI(ratio, "F1") + "x");
}
display(sbEff.ToString());
display("Conclusion : l'ecart d'efficacite SPE vs cooperation CROIT avec le nombre de tours.");
display("C'est pourquoi le paradoxe est si marque dans les versions longues du mille-pattes.");

Efficacite SPE vs Cooperation (Centipede) :
--------------------------------------------------------
  Tours |  SPE (Take t=0) | Cooperatif (fin) | Ratio
  00002 |  1.0          |  2.0            | 2.0x
  00004 |  1.0          |  4.0            | 4.0x
  00006 |  1.0          |  6.0            | 6.0x
  00008 |  1.0          |  8.0            | 8.0x
  00010 |  1.0          |  10.0            | 10.0x


Conclusion : l'ecart d'efficacite SPE vs cooperation CROIT avec le nombre de tours.

C'est pourquoi le paradoxe est si marque dans les versions longues du mille-pattes.

## 6. Guerre d'usure (War of Attrition)

Deux joueurs s'affrontent ; a chaque tour chacun peut `Quit` (abandonner) ou `Fight` (continuer).
Le gagnant empoche un prix **V**, mais chaque tour de combat coute **c** aux deux joueurs.

**Tension** : abandonner vite economise les couts, mais si l'autre abandonne d'abord, on gagne V.

In [6]:
// Guerre d'usure sequentielle (simplifiee) : J1 puis J2 decident a chaque tour.
public static ExtensiveFormGame CreateWarOfAttrition(int maxRounds = 5, double V = 10, double c = 1)
{
    var game = new ExtensiveFormGame("War of Attrition (V=" + FI(V, "F0") + ", c=" + FI(c, "F0") + ")", 2);

    GameNode BuildRound(int round, double cost1, double cost2)
    {
        if (round >= maxRounds)
        {
            var tie = new GameNode("tie_" + round, -1);
            tie.Payoffs = new double[]{ -cost1, -cost2 };
            game.AddNode(tie);
            return tie;
        }
        // J1 abandonne : J2 gagne V.
        var j1Quit = new GameNode("j1_quit_" + round, -1);
        j1Quit.Payoffs = new double[]{ -cost1, V - cost2 };
        game.AddNode(j1Quit);
        // J2 abandonne (si J1 continue) : J1 gagne V, mais a paye un c supplementaire.
        var j2Quit = new GameNode("j2_quit_" + round, -1);
        j2Quit.Payoffs = new double[]{ V - cost1 - c, -cost2 };
        game.AddNode(j2Quit);
        // Tour suivant (les deux continuent).
        var nextRound = BuildRound(round + 1, cost1 + c, cost2 + c);

        var j2Node = new GameNode("j2_" + round, 2);
        j2Node.Actions = new List<string>{"Quit", "Fight"};
        j2Node.Children["Quit"] = j2Quit;
        j2Node.Children["Fight"] = nextRound;
        j2Node.Infoset = "I2_" + round;
        game.AddNode(j2Node);

        var j1Node = new GameNode("j1_" + round, 1);
        j1Node.Actions = new List<string>{"Quit", "Fight"};
        j1Node.Children["Quit"] = j1Quit;
        j1Node.Children["Fight"] = j2Node;
        j1Node.Infoset = "I1_" + round;
        game.AddNode(j1Node);
        return j1Node;
    }

    var root = BuildRound(0, 0, 0);
    game.SetRoot(root);
    return game;
}

var woa = CreateWarOfAttrition(5, 10, 1);
var (woaSolution, woaEq) = BackwardInduction(woa);
DisplayBackwardInduction(woa, woaSolution);
display("Gains a l'equilibre : " + FV(woaEq));
display("Analyse : au dernier tour possible, abandonner domine ; par induction,");
display("la guerre d'usure se termine tot. Le ratio V/c determine la duree d'escalade.");

Solution par induction arriere : War of Attrition (V=10, c=1)
  Noeud j2_4 (J2): joue 'Quit' -> [5.00, -4.00]
  Noeud j1_4 (J1): joue 'Fight' -> [5.00, -4.00]
  Noeud j2_3 (J2): joue 'Quit' -> [6.00, -3.00]
  Noeud j1_3 (J1): joue 'Fight' -> [6.00, -3.00]
  Noeud j2_2 (J2): joue 'Quit' -> [7.00, -2.00]
  Noeud j1_2 (J1): joue 'Fight' -> [7.00, -2.00]
  Noeud j2_1 (J2): joue 'Quit' -> [8.00, -1.00]
  Noeud j1_1 (J1): joue 'Fight' -> [8.00, -1.00]
  Noeud j2_0 (J2): joue 'Quit' -> [9.00, -0.00]
  Noeud j1_0 (J1): joue 'Fight' -> [9.00, -0.00]


Gains a l'equilibre : [9.00, -0.00]

Analyse : au dernier tour possible, abandonner domine ; par induction,

la guerre d'usure se termine tot. Le ratio V/c determine la duree d'escalade.

## 7. Paradoxe de la chaine de magasins (Chain Store)

Un monopole (J2) fait face a `n` entrants **sequentiellement**. Pour chaque entrant, le monopole peut `Fight` (agressif, cout -1 pour les deux) ou `Accommodate` (+1 pour les deux). Si le monopole combat, cela dissuade... en theorie.

**Paradoxe de Selten** : l'induction arriere dit "Accommodate partout" (le dernier entrant n'a rien a craindre, donc l'avant-dernier non plus, etc.), mais intuitivement le monopole a interet a construire une reputation agressive.

In [7]:
// Chaine de magasins : n entrants sequentiels, monopole Fight/Accommodate a chaque fois.
public static ExtensiveFormGame CreateChainStoreGame(int nEntrants = 3)
{
    var game = new ExtensiveFormGame("Chain Store (" + nEntrants + " entrants)", 2);

    GameNode BuildMarket(int market, double monopolyTotal)
    {
        if (market >= nEntrants)
        {
            var end = new GameNode("end", -1);
            end.Payoffs = new double[]{ 0, monopolyTotal };
            game.AddNode(end);
            return end;
        }
        // Monopole combat sur ce marche.
        var fightContinue = BuildMarket(market + 1, monopolyTotal - 1);
        var fightTerminal = new GameNode("fight_" + market, -1);
        fightTerminal.Payoffs = new double[]{ -1, -1 };
        game.AddNode(fightTerminal);
        // Monopole accepte sur ce marche.
        var accContinue = BuildMarket(market + 1, monopolyTotal + 1);

        var monopole = new GameNode("monopole_" + market, 2);
        monopole.Actions = new List<string>{"Fight", "Accommodate"};
        monopole.Children["Fight"] = fightContinue;
        monopole.Children["Accommodate"] = accContinue;
        monopole.Infoset = "I2_" + market;
        game.AddNode(monopole);

        var entrant = new GameNode("entrant_" + market, 1);
        entrant.Actions = new List<string>{"Stay Out", "Enter"};
        entrant.Children["Stay Out"] = BuildMarket(market + 1, monopolyTotal + 2);
        entrant.Children["Enter"] = monopole;
        game.AddNode(entrant);
        return entrant;
    }

    var root = BuildMarket(0, 0);
    game.SetRoot(root);
    return game;
}

var cs = CreateChainStoreGame(3);
var (csSolution, csEq) = BackwardInduction(cs);
display("Chain Store (3 entrants) - solution synthetique :");
display("  Gains d'equilibre SPE : " + FV(csEq));
int accommodateCount = csSolution.Values.Count(s => s.OptimalAction == "Accommodate");
int fightCount = csSolution.Values.Count(s => s.OptimalAction == "Fight");
display("  Monopole : Accommodate=" + accommodateCount + " fois, Fight=" + fightCount + " fois");
display("Paradoxe : l'induction arriere pousse le monopole a Accommoder a chaque marche.");
display("Une analyse 'forward' suggere pourtant qu'une reputation agressive serait rationnelle.");

Chain Store (3 entrants) - solution synthetique :

  Gains d'equilibre SPE : [0.00, 6.00]

  Monopole : Accommodate=3 fois, Fight=0 fois

Paradoxe : l'induction arriere pousse le monopole a Accommoder a chaque marche.

Une analyse 'forward' suggere pourtant qu'une reputation agressive serait rationnelle.

## 8. Visualisation ASCII de l'arbre de jeu

Le notebook Python utilise `matplotlib` ; ce twin C# dessine l'arbre en **ASCII** (complementaire, Prong B). L'equilibre SPE est marque par une astérisque `*`.

In [8]:
// Dessine l'arbre de jeu en ASCII avec le chemin SPE marque.
public static void DrawTreeAscii(ExtensiveFormGame game, Dictionary<string, NodeSolution> solution)
{
    var sb = new StringBuilder();
    void Rec(GameNode node, string indent, bool isLast, bool onPath)
    {
        string marker = onPath ? " *" : "  ";
        string branch = (indent.Length == 0) ? "" : (isLast ? " +-- " : " |-- ");
        string label;
        if (node.IsTerminal())
            label = "[" + node.NodeId + "] = " + FV(node.Payoffs);
        else if (node.IsChance())
            label = "(" + node.NodeId + ", Nature)";
        else
            label = "(" + node.NodeId + ", J" + node.Player + ")";
        sb.AppendLine(indent + branch + marker + label);

        string childIndent = indent + (isLast ? "     " : " |    ");
        string optAction = null;
        if (solution.ContainsKey(node.NodeId)) optAction = solution[node.NodeId].OptimalAction;
        int n = node.Actions.Count;
        for (int i = 0; i < n; i++)
        {
            var action = node.Actions[i];
            var child = node.Children[action];
            bool childOnPath = onPath && (action == optAction);
            // edge label
            string edgeMark = (action == optAction) ? " *" : "  ";
            sb.AppendLine(childIndent + (i == n - 1 ? " +-- " : " |-- ") + edgeMark + "[" + action + "]");
            string grandIndent = childIndent + (i == n - 1 ? "     " : " |    ");
            Rec(child, grandIndent, true, childOnPath);
        }
    }
    sb.AppendLine("Arbre de jeu '" + game.Name + "' (* = chemin SPE) :");
    Rec(game.Root, "", true, true);
    display(sb.ToString());
}

DrawTreeAscii(entryGame, entrySolution);

Arbre de jeu 'Entry Game' (* = chemin SPE) :
 *(entrant, J1)
      |--  *[Enter]
      |     +--  *(incumbent, J2)
      |          |--   [Fight]
      |          |     +--   [fight] = [-1.00, -1.00]
      |          +--  *[Accommodate]
      |               +--  *[accommodate] = [1.00, 1.00]
      +--   [Out]
           +--   [out] = [0.00, 2.00]


## 9. Resume

### Points cles
- L'**induction arriere** resout exactement les jeux a **information parfaite** en remontant des feuilles vers la racine.
- Elle produit un **equilibre de sous-jeu parfait** (SPE) : rationnel dans chaque sous-jeu, sans menaces non credibles.
- Les **paradoxes** (centipede, chain store) montrent la tension entre cette rationalite "backward" et l'intuition "forward" (reputation, cooperation).

### Comparaison avec le twin Python
| Aspect | Python | C# (ce notebook) |
|--------|--------|-------------------|
| Structure | `@dataclass` + numpy | `class` mutable + `double[]` |
| Recursion | `def solve(node)` interne | `double[] Solve(GameNode)` locale |
| Visualisation | `matplotlib` (graphe) | ASCII tree (complementaire) |
| Dependance | numpy | BCL seule (0 NuGet) |

### Applications
- Theorie des jeux extensive (negociations, duree, reputation)
- Economie (entry deterrence, price wars)
- IA : planification adversariale, jeu a somme nulle parfait

## 10. Exercices

> **Convention** : ces cellules sont des **stubs a completer** (regle C.1 : pas d'erreur volontaire).
> Le notebook s'execute de bout en bout meme si les exercices ne sont pas resolus.

### Exercice 1 : Jeu de negociation (bargaining) a 3 tours

Deux joueurs se partagent un gain de 4. A chaque tour, celui qui propose fait une offre (x, 4-x) ; l'autre accepte ou refuse. Si refus, le gain **diminue** (discount 0.5 par tour). Apres 3 tours, le jeu s'arrete.

**Objectif** : construire l'arbre, resoudre par induction arriere, identifier l'offre d'equilibre.

In [9]:
// Exercice 1 : Bargaining a 3 tours (a completer)
// Etape 1 : construire l'arbre ExtensiveFormGame avec discount 0.5 par tour refuse
// Etape 2 : appeler BackwardInduction et afficher l'offre d'equilibre
// Indice : au tour 3 (dernier), l'offreur prend tout le reste ; remontez avec le discount.
// var bargaining = ... ;
// var (bSol, bEq) = BackwardInduction(bargaining);
// DisplayBackwardInduction(bargaining, bSol);
display("Exercice 1 (bargaining) a completer : voir indice ci-dessus.");

Exercice 1 (bargaining) a completer : voir indice ci-dessus.

### Exercice 2 : Jeu de l'ultimatum

Le proposeur offre une part x de 10 ; le recepteur accepte (10-x, x) ou refuse (0, 0).

**Objectif** : determiner l'offre SPE. Quel x le proposeur choisit-il a l'equilibre ?

In [10]:
// Exercice 2 : Jeu de l'ultimatum (a completer)
// Etape 1 : construire l'arbre (proposeur offre parmi {1, 2, ..., 5}, recepteur Accept/Reject)
// Etape 2 : resoudre par induction arriere
// Indice : le recepteur accepte toujours x > 0 (x > 0 > 0 du rejet) ; le proposeur offre donc le minimum.
// var ultimatum = ... ;
display("Exercice 2 (ultimatum) a completer : construire l'arbre puis BackwardInduction.");

Exercice 2 (ultimatum) a completer : construire l'arbre puis BackwardInduction.

### Exercice 3 : Take-Away (Nim a 1 tas)

`n` jetons sur la table. 2 joueurs retirent a tour de role 1 a `max_take` jetons. Celui qui prend le **dernier** gagne.

**Objectif** : modeliser comme jeu extensive (arbre), resoudre par induction, identifier les positions gagnantes.

In [11]:
// Exercice 3 : Take-Away / Nim a 1 tas (a completer)
// Etape 1 : ecrire une fonction buildTakeAway(n, maxTake) -> ExtensiveFormGame
// Etape 2 : resoudre et determiner pour quels n le joueur 1 gagne
// Indice : les positions perdantes sont les multiples de (maxTake + 1).
// public static ExtensiveFormGame BuildTakeAway(int n, int maxTake) { ... }
display("Exercice 3 (Take-Away) a completer : modeliser l'arbre puis BackwardInduction.");

Exercice 3 (Take-Away) a completer : modeliser l'arbre puis BackwardInduction.

---
**See** #4956 (marathon parite .NET), #3801 (Prong B : from-scratch), #2161 (exercices).
**Suite** : [GameTheory-11-BayesianGames-Csharp](GameTheory-11-BayesianGames-Csharp.ipynb) (jeux bayesiens).